# Imports

In [1]:
import numpy as np
import pandas as pd 

# Extract Dataset

In [2]:
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Transform

## change sentiment to numeric

In [3]:
df['sentiment'] = df['sentiment'].map({
    "positive": 1,
    "negative": 0
})

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [5]:
df['sentiment'].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

## convert to HuggingFace Dataset

In [6]:
from datasets import Dataset

In [7]:
dataset = Dataset.from_pandas(df)

In [8]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1
...,...,...
49995,I thought this movie did a down right good job...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",0
49997,I am a Catholic taught in parochial elementary...,0
49998,I'm going to have to disagree with the previou...,0


In [9]:
dataset

Dataset({
    features: ['review', 'sentiment'],
    num_rows: 50000
})

# helper method

In [10]:
def show_demo(dataset):
    print(dataset['train'][0])

# Train/Test Split

In [11]:
dataset = dataset.train_test_split(test_size=0.2)

In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['review', 'sentiment'],
        num_rows: 10000
    })
})

In [13]:
show_demo(dataset)

{'review': 'This film for me and my wife is more entertaining than all the bloc-buster violent thriller/mystery/murder movies that abound. It is about real people making the best of their lives. They just happen to be Indian and the main characters are in law enforcement. The realistic acting and the great scenery more than make up for the slightly implausible plot. The sound track is by BC Smith, who also did the soundtrack for Coyote Waits, and is great. Adam Beach plays a tribal policeman who is a little bit accident prone and Wes Studi is the stoic consummately professional detective. There are many other fine either supporting or cameo roles by Graham Greene, Tantoo Cardinal, etc. We have also seen Coyote Waits, another adaptation of a Hillerman novel, and we greatly enjoyed it too.', 'sentiment': 1}


# Tokenization

In [14]:
from transformers import AutoTokenizer

In [15]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
def tokenize(case):
    return tokenizer(
        case['review'],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [17]:
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [18]:
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['review', 'sentiment', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10000
    })
})

In [19]:
show_demo(dataset)

{'review': 'This film for me and my wife is more entertaining than all the bloc-buster violent thriller/mystery/murder movies that abound. It is about real people making the best of their lives. They just happen to be Indian and the main characters are in law enforcement. The realistic acting and the great scenery more than make up for the slightly implausible plot. The sound track is by BC Smith, who also did the soundtrack for Coyote Waits, and is great. Adam Beach plays a tribal policeman who is a little bit accident prone and Wes Studi is the stoic consummately professional detective. There are many other fine either supporting or cameo roles by Graham Greene, Tantoo Cardinal, etc. We have also seen Coyote Waits, another adaptation of a Hillerman novel, and we greatly enjoyed it too.', 'sentiment': 1, 'input_ids': [101, 2023, 2143, 2005, 2033, 1998, 2026, 2564, 2003, 2062, 14036, 2084, 2035, 1996, 15984, 1011, 18396, 6355, 10874, 1013, 6547, 1013, 4028, 5691, 2008, 11113, 28819, 10

# setting the format for Pytorch

In [20]:
dataset = dataset.rename_column("sentiment","labels")

In [21]:
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['review', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10000
    })
})

In [22]:
show_demo(dataset)

{'review': 'This film for me and my wife is more entertaining than all the bloc-buster violent thriller/mystery/murder movies that abound. It is about real people making the best of their lives. They just happen to be Indian and the main characters are in law enforcement. The realistic acting and the great scenery more than make up for the slightly implausible plot. The sound track is by BC Smith, who also did the soundtrack for Coyote Waits, and is great. Adam Beach plays a tribal policeman who is a little bit accident prone and Wes Studi is the stoic consummately professional detective. There are many other fine either supporting or cameo roles by Graham Greene, Tantoo Cardinal, etc. We have also seen Coyote Waits, another adaptation of a Hillerman novel, and we greatly enjoyed it too.', 'labels': 1, 'input_ids': [101, 2023, 2143, 2005, 2033, 1998, 2026, 2564, 2003, 2062, 14036, 2084, 2035, 1996, 15984, 1011, 18396, 6355, 10874, 1013, 6547, 1013, 4028, 5691, 2008, 11113, 28819, 1012,

In [23]:
dataset.set_format(type="torch", columns=['input_ids','attention_mask','labels'])

In [24]:
dataset

DatasetDict({
    train: Dataset({
        features: ['review', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['review', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10000
    })
})

In [25]:
show_demo(dataset)

{'labels': tensor(1), 'input_ids': tensor([  101,  2023,  2143,  2005,  2033,  1998,  2026,  2564,  2003,  2062,
        14036,  2084,  2035,  1996, 15984,  1011, 18396,  6355, 10874,  1013,
         6547,  1013,  4028,  5691,  2008, 11113, 28819,  1012,  2009,  2003,
         2055,  2613,  2111,  2437,  1996,  2190,  1997,  2037,  3268,  1012,
         2027,  2074,  4148,  2000,  2022,  2796,  1998,  1996,  2364,  3494,
         2024,  1999,  2375,  7285,  1012,  1996, 12689,  3772,  1998,  1996,
         2307, 17363,  2062,  2084,  2191,  2039,  2005,  1996,  3621, 17727,
        28128,  7028,  5436,  1012,  1996,  2614,  2650,  2003,  2011,  4647,
         3044,  1010,  2040,  2036,  2106,  1996,  6050,  2005, 20457, 18074,
         1010,  1998,  2003,  2307,  1012,  4205,  3509,  3248,  1037,  8807,
        14460,  2040,  2003,  1037,  2210,  2978,  4926, 13047,  1998, 14008,
        16054,  2072,  2003,  1996,  2358, 19419,  9530, 17421,  8585,  2135,
         2658,  6317,  1012, 

# Model

In [26]:
from transformers import AutoModelForSequenceClassification

In [27]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Train

## Metrics

In [28]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

## Training

In [29]:
from transformers import TrainingArguments, Trainer

In [30]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=8,
    num_train_epochs=2
)

In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    compute_metrics=compute_metrics
)

In [32]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
500,0.738381
1000,0.558757
1500,0.509103
2000,0.479682
2500,0.460785
3000,0.317629
3500,0.301585
4000,0.305003
4500,0.293315
5000,0.258918


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5000, training_loss=0.4223158203125, metrics={'train_runtime': 2409.1954, 'train_samples_per_second': 33.206, 'train_steps_per_second': 2.075, 'total_flos': 1.05244422144e+16, 'train_loss': 0.4223158203125, 'epoch': 2.0})

# Evaluate

In [33]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.5493413209915161,
 'eval_accuracy': 0.9306,
 'eval_runtime': 100.4947,
 'eval_samples_per_second': 99.508,
 'eval_steps_per_second': 6.219,
 'epoch': 2.0}